# STC Jawwy

In [4]:
"""
Here we install libraries that are not installed by default 
Example:  pyslsb
Feel free to add any library you are planning to use.
"""
!pip install pyxlsb

In [5]:
# Import the required libraries 
"""
Please feel free to import any required libraries as per your needs
"""
import pandas as pd     # provides high-performance, easy to use structures and data analysis tools
import pyxlsb           # Excel extention to read xlsb files (the input file)
import numpy as np      # provides fast mathematical computation on arrays and matrices

# Jawwy dataset
The dataset consists of details about each customer and the movies and/or tv shows watched in addition to the genre. 

You are required to work on task three to build a recommendation engine for our platform to Recommend movies to usesrs that they might be interested in¶


In [6]:
dataframe = pd.read_excel("stc TV Data Set_T3.xlsx",index_col=0)
# Please make a copy of dataset if you are going to work directly and make changes on the dataset
# you can use   df=dataframe.copy()

In [7]:
# check the data shape
dataframe.shape

(1048575, 5)

In [8]:
# display the first 5 rows 
dataframe.head()

,user_id_maped,program_name,rating,date_,program_genre
0,26138,100 treets,1,2017-05-27,Drama
1,7946,Moana,1,2017-05-21,Animation
2,7418,The Mermaid Princess,1,2017-08-10,Animation
3,19307,The Mermaid Princess,2,2017-07-26,Animation
4,15860,Churchill,2,2017-07-07,Biography


In [9]:
# describe the numeric values in the dataset
dataframe.describe()

,user_id_maped,rating,date_
count,1.048575e+06,1.048575e+06,1048575
mean,1.709266e+04,2.497283e+00,2017-10-04 00:23:20.346183
min,1.000000e+00,1.000000e+00,2017-03-14 00:00:00
25%,8.253000e+03,1.000000e+00,2017-06-10 00:00:00
50%,1.714900e+04,2.000000e+00,2017-10-14 00:00:00
75%,2.566500e+04,3.000000e+00,2018-01-21 00:00:00
max,3.428000e+04,4.000000e+00,2018-04-30 00:00:00
std,1.003513e+04,1.119837e+00,NaN


In [10]:
# check if any column has null value in the dataset
dataframe.isnull().any()

user_id_maped    False
program_name     False
rating           False
date_            False
program_genre    False
dtype: bool

In [11]:
# we import Visualization libraries 
# you can ignore and use any other graphing libraries 
import matplotlib.pyplot as plt # a comprehensive library for creating static, animated, and interactive visualizations
import plotly #a graphing library makes interactive, publication-quality graphs. Examples of how to make line plots, scatter plots, area charts, bar charts, error bars, box plots, histograms, heatmaps, subplots, multiple-axes, polar charts, and bubble charts.
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [74]:
"""
TODO build your Recommender system to Highlight Programs that usesrs might be interested in
"""
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix


In [ ]:
# 1. Clean the data (Use the full dataframe)
df_final = dataframe.drop_duplicates(subset=['user_id_maped', 'program_name'])

# 2. Create the Matrix (Rows = Movies, Columns = ALL Users)
# This will create exactly 11,578 features (columns)
pivot_full = df_final.pivot(index='program_name', 
                            columns='user_id_maped', 
                            values='rating').fillna(0)

# 3. Convert to Sparse
matrix_sparse_full = csr_matrix(pivot_full.values)

# 4. Fit the model
# We use 'full' in the name so it doesn't conflict with your previous test
model_knn_full = NearestNeighbors(metric='cosine', algorithm='brute')
model_knn_full.fit(matrix_sparse_full)

print(f"Model Ready! It is now expecting {pivot_full.shape[1]} users.")

Model Ready! It is now expecting 11578 users.


In [78]:
def get_final_recommendations(user_id, n=5):
    # Get user history
    user_history = df_final[df_final['user_id_maped'] == user_id]['program_name'].unique()
    
    recs = []
    for movie in user_history:
        try:
            # Get the row index for the movie
            idx = pivot_full.index.get_loc(movie)
            
            # Find neighbors using the FULL model
            distances, indices = model_knn_full.kneighbors(matrix_sparse_full[idx], n_neighbors=6)
            
            for i in range(1, len(indices.flatten())):
                sim_movie = pivot_full.index[indices.flatten()[i]]
                if sim_movie not in user_history:
                    recs.append(sim_movie)
        except:
            continue
            
    return list(set(recs))[:n]

In [79]:
"""
TODO show the recommendations (top 5) for the people who watched "Moana" movie
"""

'\nTODO show the recommendations (top 5) for the people who watched "Moana" movie\n'

In [80]:
test_movie = "Moana"
movie_idx = pivot_full.index.get_loc(test_movie)
distances, indices = model_knn_full.kneighbors(matrix_sparse_full[movie_idx], n_neighbors=6)

print(f"Top 5 recommendations from viewers who like '{test_movie}': ")
for i in range(1, 6):
    print(f"{i}. {pivot_full.index[indices.flatten()[i]]}")


Top 5 recommendations from viewers who like 'Moana': 
1. Trolls
2. Surf's Up : WaveMania
3. The Mermaid Princess
4. The Boss Baby
5. The Jetsons & WWE: Robo-WrestleMania!
